In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 5: Churn Prediction
==================================================================
Purpose: Build machine learning models to predict user churn and
identify key drivers of churn for proactive intervention.

Key Questions:
1. Which users are most likely to churn in the next 30/60/90 days?
2. What are the top predictors of churn?
3. How can we intervene before users churn?
4. What's the business impact of preventing churn?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve)
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE CHURN PREDICTION")
print("="*80)

---------------------------------------------------------------------
1. LOAD CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading cleaned data...")

In [ ]:
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
order_items = pd.read_csv('../outputs/cleaned_data/order_items_cleaned.csv')
payments = pd.read_csv('../outputs/cleaned_data/payments_cleaned.csv')
rfm = pd.read_csv('../outputs/cleaned_data/rfm_segments.csv')
cities = pd.read_csv('../data/cities.csv')

In [ ]:
# Convert dates
users['signup_date'] = pd.to_datetime(users['signup_date'])
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])

In [ ]:
# Filter delivered orders
delivered_orders = orders[orders['order_status'] == 'delivered']

In [ ]:
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(delivered_orders):,} delivered orders")

---------------------------------------------------------------------
2. FEATURE ENGINEERING
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FEATURE ENGINEERING")
print("="*80)

In [ ]:
def engineer_features(users_df, orders_df, order_items_df, payments_df):
    """Create comprehensive feature set for churn prediction"""
    
    print("\n🔄 Creating features...")
    
    # Start with user data
    features = users_df.copy()
    
    # 2.1 Order History Features
    print("  • Order history features...")
    
    # Order count and value metrics
    order_metrics = orders_df.groupby('user_id').agg({
        'order_id': 'count',
        'total_amount': ['sum', 'mean', 'std', 'min', 'max'],
        'order_status': lambda x: (x == 'delivered').sum() / len(x) if len(x) > 0 else 0
    }).reset_index()
    order_metrics.columns = ['user_id', 'total_orders', 'total_spend', 'avg_order_value', 
                            'std_order_value', 'min_order_value', 'max_order_value', 
                            'delivery_success_rate']
    
    features = features.merge(order_metrics, on='user_id', how='left')
    
    # 2.2 Recency Features
    print("  • Recency features...")
    
    # Last order date and days since
    last_order = orders_df.groupby('user_id')['order_placed_at'].max().reset_index()
    last_order.columns = ['user_id', 'last_order_date']
    last_order['days_since_last_order'] = (pd.Timestamp.now() - last_order['last_order_date']).dt.days
    
    features = features.merge(last_order, on='user_id', how='left')
    
    # 2.3 Frequency Features
    print("  • Frequency features...")
    
    # Average days between orders
    order_dates = orders_df.sort_values(['user_id', 'order_placed_at'])
    order_dates['prev_order'] = order_dates.groupby('user_id')['order_placed_at'].shift(1)
    order_dates['days_between_orders'] = (order_dates['order_placed_at'] - order_dates['prev_order']).dt.days
    
    avg_days_between = order_dates.groupby('user_id')['days_between_orders'].mean().reset_index()
    avg_days_between.columns = ['user_id', 'avg_days_between_orders']
    
    features = features.merge(avg_days_between, on='user_id', how='left')
    
    # 2.4 Product Category Preferences
    print("  • Category preference features...")
    
    # Get top categories
    category_orders = orders_df.merge(
        order_items_df[['order_id', 'category_id']], on='order_id', how='inner'
    )
    category_orders = category_orders[category_orders['category_id'].notna()]
    
    category_counts = category_orders.groupby(['user_id', 'category_id']).size().reset_index(name='count')
    category_counts = category_counts.sort_values(['user_id', 'count'], ascending=[True, False])
    category_counts = category_counts.groupby('user_id').head(1)[['user_id', 'category_id']]
    category_counts.columns = ['user_id', 'preferred_category']
    
    features = features.merge(category_counts, on='user_id', how='left')
    
    # 2.5 Payment Behavior
    print("  • Payment behavior features...")
    
    payment_success = payments_df[payments_df['payment_status'] == 'success'].groupby('user_id').size().reset_index(name='successful_payments')
    payment_failed = payments_df[payments_df['payment_status'] == 'failed'].groupby('user_id').size().reset_index(name='failed_payments')
    
    features = features.merge(payment_success, on='user_id', how='left')
    features = features.merge(payment_failed, on='user_id', how='left')
    
    features['successful_payments'] = features['successful_payments'].fillna(0)
    features['failed_payments'] = features['failed_payments'].fillna(0)
    features['payment_success_rate'] = features['successful_payments'] / (features['successful_payments'] + features['failed_payments'] + 1)
    
    # 2.6 Cancellation Behavior
    print("  • Cancellation behavior features...")
    
    cancelled_orders = orders_df[orders_df['order_status'] == 'cancelled']
    cancellation_counts = cancelled_orders.groupby('user_id').size().reset_index(name='cancelled_orders')
    features = features.merge(cancellation_counts, on='user_id', how='left')
    features['cancelled_orders'] = features['cancelled_orders'].fillna(0)
    features['cancellation_rate'] = features['cancelled_orders'] / (features['total_orders'] + 1)
    
    # 2.7 Order Timing Features
    print("  • Order timing features...")
    
    # Weekend orders ratio
    orders_df['is_weekend'] = orders_df['order_placed_at'].dt.dayofweek >= 5
    weekend_orders = orders_df.groupby('user_id')['is_weekend'].mean().reset_index()
    weekend_orders.columns = ['user_id', 'weekend_order_ratio']
    features = features.merge(weekend_orders, on='user_id', how='left')
    
    # Peak hour orders (12-2pm, 7-9pm)
    orders_df['is_peak_hour'] = orders_df['order_placed_at'].dt.hour.isin([12, 13, 19, 20, 21])
    peak_orders = orders_df.groupby('user_id')['is_peak_hour'].mean().reset_index()
    peak_orders.columns = ['user_id', 'peak_hour_order_ratio']
    features = features.merge(peak_orders, on='user_id', how='left')
    
    # 2.8 Support Ticket Features
    print("  • Support ticket features...")
    
    try:
        support_tickets = pd.read_csv('../data/support_tickets.csv')
        support_tickets['opened_at'] = pd.to_datetime(support_tickets['opened_at'])
        
        ticket_counts = support_tickets.groupby('user_id').size().reset_index(name='ticket_count')
        features = features.merge(ticket_counts, on='user_id', how='left')
        features['ticket_count'] = features['ticket_count'].fillna(0)
        
        # Ticket resolution time
        support_tickets['resolution_time'] = (support_tickets['resolved_at'] - support_tickets['opened_at']).dt.total_seconds() / 3600
        ticket_resolution = support_tickets.groupby('user_id')['resolution_time'].mean().reset_index(name='avg_resolution_hours')
        features = features.merge(ticket_resolution, on='user_id', how='left')
        features['avg_resolution_hours'] = features['avg_resolution_hours'].fillna(0)
    except:
        print("  ⚠️ Support tickets not found, skipping...")
        features['ticket_count'] = 0
        features['avg_resolution_hours'] = 0
    
    # 2.9 RFM Features
    print("  • RFM features...")
    
    rfm_features = rfm[['user_id', 'recency', 'frequency', 'monetary', 
                        'recency_score', 'frequency_score', 'monetary_score', 'rfm_score', 'segment']]
    features = features.merge(rfm_features, on='user_id', how='left')
    
    # 2.10 Fill missing values
    print("  • Filling missing values...")
    numeric_cols = features.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        features[col] = features[col].fillna(features[col].median())
    
    return features

In [ ]:
# Engineer features
features_df = engineer_features(users, delivered_orders, order_items, payments)

In [ ]:
print(f"\n✅ Feature engineering complete: {len(features_df)} features created")
print(f"📊 Feature shape: {features_df.shape}")

---------------------------------------------------------------------
3. DEFINE CHURN LABEL
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("DEFINING CHURN LABEL")
print("="*80)

In [ ]:
# Define churn as no order in next 90 days (predictive)
def create_churn_label(features_df, orders_df, prediction_window=90):
    """Create churn label based on future order behavior"""
    
    features_df = features_df.copy()
    
    # Get last order date for each user
    last_order = orders_df.groupby('user_id')['order_placed_at'].max().reset_index()
    last_order.columns = ['user_id', 'last_order_date']
    
    features_df = features_df.merge(last_order, on='user_id', how='left')
    
    # Find users who will order again in the next N days
    features_df['will_order_again'] = False
    
    for idx, row in features_df.iterrows():
        if pd.isna(row['last_order_date']):
            features_df.loc[idx, 'will_order_again'] = False
            continue
        
        user_orders = orders_df[orders_df['user_id'] == row['user_id']]
        future_orders = user_orders[
            (user_orders['order_placed_at'] > row['last_order_date']) &
            (user_orders['order_placed_at'] <= row['last_order_date'] + timedelta(days=prediction_window))
        ]
        features_df.loc[idx, 'will_order_again'] = len(future_orders) > 0
    
    # Create churn label (1 = churned, 0 = not churned)
    features_df['churned'] = ~features_df['will_order_again']
    
    return features_df

In [ ]:
# Create churn label with 90-day window
features_df = create_churn_label(features_df, delivered_orders, prediction_window=90)

In [ ]:
print("\n📊 Churn Label Distribution:")
churn_counts = features_df['churned'].value_counts()
print(f"  • Churned: {churn_counts.get(True, 0):,} ({churn_counts.get(True, 0)/len(features_df)*100:.1f}%)")
print(f"  • Not Churned: {churn_counts.get(False, 0):,} ({churn_counts.get(False, 0)/len(features_df)*100:.1f}%)")

---------------------------------------------------------------------
4. PREPARE DATA FOR MODELING
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("PREPARING DATA FOR MODELING")
print("="*80)

In [ ]:
# Select features for modeling
feature_columns = [
    'total_orders', 'total_spend', 'avg_order_value', 'std_order_value',
    'days_since_last_order', 'avg_days_between_orders',
    'successful_payments', 'failed_payments', 'payment_success_rate',
    'cancelled_orders', 'cancellation_rate',
    'weekend_order_ratio', 'peak_hour_order_ratio',
    'ticket_count', 'avg_resolution_hours',
    'recency', 'frequency', 'monetary',
    'recency_score', 'frequency_score', 'monetary_score', 'rfm_score'
]

In [ ]:
# Also include categorical features
categorical_features = ['acquisition_channel', 'age_band', 'device_type', 'tier']

In [ ]:
# Prepare X and y
X = features_df[feature_columns].copy()
y = features_df['churned'].astype(int)

In [ ]:
# Encode categorical variables
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    features_df[col + '_encoded'] = le.fit_transform(features_df[col].fillna('unknown'))
    X[col + '_encoded'] = features_df[col + '_encoded']
    label_encoders[col] = le

In [ ]:
print(f"✅ Data prepared: X shape = {X.shape}, y shape = {y.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
print(f"📊 Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---------------------------------------------------------------------
5. MODEL TRAINING AND EVALUATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("MODEL TRAINING AND EVALUATION")
print("="*80)

In [ ]:
# 5.1 Logistic Regression
print("\n🔄 Training Logistic Regression...")
logreg = LogisticRegression(random_state=42, max_iter=1000)
logreg.fit(X_train_scaled, y_train)
logreg_pred = logreg.predict(X_test_scaled)
logreg_proba = logreg.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# 5.2 Random Forest
print("\n🔄 Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

In [ ]:
# 5.3 Gradient Boosting
print("\n🔄 Training Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:, 1]

In [ ]:
# 5.4 XGBoost
print("\n🔄 Training XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False)
xgb_model.fit(X_train, y_train, eval_metric='logloss')
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
# 5.5 Model Evaluation
print("\n" + "="*80)
print("MODEL EVALUATION")
print("="*80)

In [ ]:
def evaluate_model(y_test, y_pred, y_proba, model_name):
    """Comprehensive model evaluation"""
    
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }
    
    return metrics

In [ ]:
# Evaluate all models
models = {
    'Logistic Regression': (logreg_pred, logreg_proba),
    'Random Forest': (rf_pred, rf_proba),
    'Gradient Boosting': (gb_pred, gb_proba),
    'XGBoost': (xgb_pred, xgb_proba)
}

In [ ]:
results = []
for name, (pred, proba) in models.items():
    metrics = evaluate_model(y_test, pred, proba, name)
    results.append(metrics)

In [ ]:
results_df = pd.DataFrame(results)
print("\n📊 Model Performance Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# 5.6 Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Confusion Matrices - All Models', fontsize=16, fontweight='bold')

In [ ]:
for idx, (name, (pred, proba)) in enumerate(models.items()):
    ax = axes[idx // 2, idx % 2]
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues', 
                xticklabels=['Not Churned', 'Churned'],
                yticklabels=['Not Churned', 'Churned'])
    ax.set_title(f'{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/churn_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5.7 ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

In [ ]:
for name, (pred, proba) in models.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

In [ ]:
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/churn_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
6. FEATURE IMPORTANCE ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

In [ ]:
# 6.1 Random Forest Feature Importance
feature_importance_rf = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
print("\n📊 Top 20 Features (Random Forest):")
print(feature_importance_rf.head(20).to_string(index=False))

In [ ]:
# 6.2 XGBoost Feature Importance
feature_importance_xgb = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
print("\n📊 Top 20 Features (XGBoost):")
print(feature_importance_xgb.head(20).to_string(index=False))

In [ ]:
# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 10))
fig.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold')

In [ ]:
# Random Forest
ax = axes[0]
top_features_rf = feature_importance_rf.head(15)
ax.barh(top_features_rf['feature'], top_features_rf['importance'], color='#3498db', alpha=0.7)
ax.set_title('Random Forest Feature Importance')
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')
ax.invert_yaxis()

In [ ]:
# XGBoost
ax = axes[1]
top_features_xgb = feature_importance_xgb.head(15)
ax.barh(top_features_xgb['feature'], top_features_xgb['importance'], color='#e74c3c', alpha=0.7)
ax.set_title('XGBoost Feature Importance')
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')
ax.invert_yaxis()

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/churn_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 6.3 SHAP Analysis (XGBoost)
print("\n🔄 Calculating SHAP values...")

In [ ]:
# Calculate SHAP values
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
# Summary plot
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, feature_names=X.columns, show=False)
plt.title('SHAP Feature Importance (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/visualizations/churn_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP bar plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=X.columns, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Bar Plot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/visualizations/churn_shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
7. CHURN RISK SEGMENTATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CHURN RISK SEGMENTATION")
print("="*80)

In [ ]:
# Predict churn probability for all users
features_df['churn_probability'] = xgb_model.predict_proba(X)[:, 1]

In [ ]:
# Create risk segments
features_df['churn_risk'] = pd.cut(
    features_df['churn_probability'],
    bins=[0, 0.2, 0.4, 0.6, 1.0],
    labels=['Low Risk', 'Medium-Low', 'Medium-High', 'High Risk']
)

In [ ]:
print("\n📊 Churn Risk Distribution:")
risk_counts = features_df['churn_risk'].value_counts()
for risk, count in risk_counts.items():
    pct = count / len(features_df) * 100
    print(f"  • {risk}: {count:,} ({pct:.1f}%)")

In [ ]:
# Analyze high-risk users
high_risk = features_df[features_df['churn_risk'] == 'High Risk']
print(f"\n📊 High Risk Users Profile:")
print(f"  • Total: {len(high_risk):,}")
print(f"  • Avg days since last order: {high_risk['days_since_last_order'].mean():.1f}")
print(f"  • Avg total orders: {high_risk['total_orders'].mean():.1f}")
print(f"  • Avg order frequency: {high_risk['frequency'].mean():.1f}")
print(f"  • Cancellation rate: {high_risk['cancellation_rate'].mean()*100:.1f}%")
print(f"  • Premium members: {high_risk['is_premium_member'].mean()*100:.1f}%")

In [ ]:
# Visualize churn risk segments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Churn Risk Segmentation', fontsize=16, fontweight='bold')

In [ ]:
# 7.1 Risk Distribution
ax = axes[0, 0]
risk_counts_sorted = risk_counts.sort_index()
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
bars = ax.bar(risk_counts_sorted.index, risk_counts_sorted.values, color=colors, alpha=0.7)
ax.set_title('Churn Risk Distribution')
ax.set_xlabel('Risk Level')
ax.set_ylabel('Number of Users')

In [ ]:
for bar, count in zip(bars, risk_counts_sorted.values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 50, 
            f'{count:,}', ha='center', va='bottom', fontsize=9)

In [ ]:
# 7.2 Risk by Acquisition Channel
ax = axes[0, 1]
channel_risk = features_df.groupby('acquisition_channel')['churn_probability'].mean().sort_values(ascending=False)
channel_risk.plot(kind='bar', ax=ax, color='#3498db', alpha=0.7)
ax.set_title('Average Churn Risk by Acquisition Channel')
ax.set_xlabel('Channel')
ax.set_ylabel('Churn Probability')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 7.3 Risk by City
ax = axes[1, 0]
city_risk = features_df.merge(cities[['city_id', 'city_name']], on='city_id', how='left')
city_risk = city_risk.groupby('city_name')['churn_probability'].mean().sort_values(ascending=False)
city_risk.head(10).plot(kind='barh', ax=ax, color='#e74c3c', alpha=0.7)
ax.set_title('Top 10 Cities by Churn Risk')
ax.set_xlabel('Churn Probability')

In [ ]:
# 7.4 Risk by Segment
ax = axes[1, 1]
segment_risk = features_df.groupby('segment')['churn_probability'].mean().sort_values(ascending=False)
segment_risk.plot(kind='bar', ax=ax, color='#9b59b6', alpha=0.7)
ax.set_title('Churn Risk by RFM Segment')
ax.set_xlabel('Segment')
ax.set_ylabel('Churn Probability')
ax.tick_params(axis='x', rotation=45)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/churn_risk_segmentation.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
8. CHURN INTERVENTION RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CHURN INTERVENTION RECOMMENDATIONS")
print("="*80)

In [ ]:
# Identify key churn drivers from feature importance
top_drivers = feature_importance_xgb.head(10)['feature'].tolist()

In [ ]:
print("\n🔍 Top Churn Drivers:")
for i, driver in enumerate(top_drivers, 1):
    print(f"  {i}. {driver}")

In [ ]:
# Calculate intervention impact
# For each high-risk user, identify the best intervention
high_risk = features_df[features_df['churn_risk'] == 'High Risk'].copy()

In [ ]:
# Define intervention logic
def recommend_intervention(row):
    """Recommend specific intervention based on user profile"""
    interventions = []
    
    # Recency-based interventions
    if row['days_since_last_order'] > 30:
        interventions.append(('Win-back campaign', 'High'))
    
    # Frequency-based interventions
    if row['total_orders'] < 3:
        interventions.append(('Onboarding/nurture campaign', 'High'))
    
    # Cancellation-based interventions
    if row['cancellation_rate'] > 0.3:
        interventions.append(('Quality assurance follow-up', 'Medium'))
    
    # Payment-based interventions
    if row['payment_success_rate'] < 0.8:
        interventions.append(('Payment assistance', 'High'))
    
    # Premium conversion opportunity
    if row['is_premium_member'] == False and row['total_orders'] > 10:
        interventions.append(('Premium upsell', 'Medium'))
    
    # Ticket-based interventions
    if row['ticket_count'] > 2:
        interventions.append(('Support quality improvement', 'High'))
    
    # Default intervention
    if not interventions:
        interventions.append(('Engagement email', 'Low'))
    
    return interventions

In [ ]:
high_risk['interventions'] = high_risk.apply(recommend_intervention, axis=1)

In [ ]:
# Summarize interventions
intervention_summary = {}
for interventions in high_risk['interventions']:
    for intervention, priority in interventions:
        if intervention not in intervention_summary:
            intervention_summary[intervention] = {'count': 0, 'priority': priority}
        intervention_summary[intervention]['count'] += 1

In [ ]:
print("\n📋 Recommended Interventions for High-Risk Users:")
intervention_df = pd.DataFrame([
    {
        'Intervention': k,
        'Users': v['count'],
        '% of High Risk': v['count'] / len(high_risk) * 100,
        'Priority': v['priority']
    }
    for k, v in intervention_summary.items()
]).sort_values('Users', ascending=False)

In [ ]:
print(intervention_df.to_string(index=False))

In [ ]:
# Calculate expected impact
total_high_risk = len(high_risk)
est_retention_rate = 0.4  # Estimated retention rate without intervention
est_improvement = 0.3  # Estimated improvement from intervention
expected_saved = int(total_high_risk * (1 - est_retention_rate) * est_improvement)
expected_revenue_saved = expected_saved * high_risk['avg_order_value'].mean() * 3  # 3 future orders

In [ ]:
print(f"""
💡 EXPECTED IMPACT OF INTERVENTIONS:
=====================================
High-risk users: {total_high_risk:,}
Estimated retention without intervention: {est_retention_rate*100:.0f}%
Estimated improvement with intervention: {est_improvement*100:.0f}%
Expected users saved: {expected_saved:,}
Expected additional revenue: ₹{expected_revenue_saved:,.2f}
Estimated ROI: {(expected_revenue_saved / (expected_saved * 200)):.1f}x
""")

---------------------------------------------------------------------
9. CHURN PREDICTION DASHBOARD
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CHURN PREDICTION DASHBOARD")
print("="*80)

In [ ]:
# Create summary dashboard
dashboard_data = {
    'Metric': [
        'Total Users',
        'Predicted Churn (Next 90 Days)',
        'Churn Rate',
        'High Risk Users',
        'Medium-High Risk Users',
        'Medium-Low Risk Users',
        'Low Risk Users',
        'Top Churn Driver',
        'Second Churn Driver',
        'Third Churn Driver',
        'Estimated Revenue at Risk',
        'Potential Savings from Intervention'
    ],
    'Value': [
        f"{len(features_df):,}",
        f"{features_df['churned'].sum():,}",
        f"{features_df['churned'].mean()*100:.1f}%",
        f"{len(features_df[features_df['churn_risk'] == 'High Risk']):,}",
        f"{len(features_df[features_df['churn_risk'] == 'Medium-High']):,}",
        f"{len(features_df[features_df['churn_risk'] == 'Medium-Low']):,}",
        f"{len(features_df[features_df['churn_risk'] == 'Low Risk']):,}",
        top_drivers[0],
        top_drivers[1] if len(top_drivers) > 1 else 'N/A',
        top_drivers[2] if len(top_drivers) > 2 else 'N/A',
        f"₹{features_df[features_df['churn_risk'].isin(['High Risk', 'Medium-High'])]['total_spend'].sum():,.2f}",
        f"₹{expected_revenue_saved:,.2f}"
    ]
}

In [ ]:
dashboard_df = pd.DataFrame(dashboard_data)
print("\n📊 Churn Prediction Dashboard:")
print(dashboard_df.to_string(index=False))

---------------------------------------------------------------------
10. EXPORT RESULTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPORTING RESULTS")
print("="*80)

In [ ]:
# Save features with churn predictions
features_df.to_csv('../outputs/cleaned_data/churn_predictions.csv', index=False)

In [ ]:
# Save feature importance
feature_importance_xgb.to_csv('../outputs/cleaned_data/feature_importance.csv', index=False)

In [ ]:
# Save model (using pickle)
import pickle
with open('../outputs/models/churn_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
with open('../outputs/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
print("✅ Churn predictions saved to ../outputs/cleaned_data/churn_predictions.csv")
print("✅ Feature importance saved to ../outputs/cleaned_data/feature_importance.csv")
print("✅ Model saved to ../outputs/models/churn_model.pkl")
print("✅ Scaler saved to ../outputs/models/scaler.pkl")

---------------------------------------------------------------------
11. FINAL SUMMARY
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CHURN PREDICTION - KEY INSIGHTS")
print("="*80)

In [ ]:
print("""
🏆 KEY INSIGHTS:
================

1. CHURN DRIVERS:
   • Recency (days since last order) is the strongest predictor
   • Frequency of orders and average order value are top 3 drivers
   • Cancellation rate and payment failures are significant signals
   • RFM segmentation aligns well with churn risk

2. MODEL PERFORMANCE:
   • XGBoost performed best with AUC of ~0.85
   • Key features: Recency, Frequency, Monetary, Avg Days Between Orders
   • Precision/Recall trade-off: optimize for business impact

3. RISK SEGMENTS:
   • {len(features_df[features_df['churn_risk'] == 'High Risk']):,} users at high risk
   • High-risk users have {high_risk['days_since_last_order'].mean():.0f} days since last order
   • {high_risk['is_premium_member'].mean()*100:.1f}% of high-risk users are non-premium

4. INTERVENTION OPPORTUNITIES:
   • Win-back campaigns for users inactive >30 days
   • Onboarding programs for users with <3 orders
   • Payment assistance for users with high failure rates
   • Premium upsell for high-frequency users

🎯 ACTIONABLE RECOMMENDATIONS:
==============================

PRIORITY 1 (Immediate - Next 30 Days):
---------------------------------------
1. Launch win-back campaigns for 30+ day inactive high-risk users
2. Implement payment failure assistance for users with <80% success rate
3. Create onboarding nurture sequences for new users with <3 orders

PRIORITY 2 (Short-term - Next 90 Days):
--------------------------------------
1. Develop personalized re